# 0.0 Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import sklearn.metrics as mt

# 0.1 - Load Dataset

In [2]:
actual_folder = Path.cwd()
main_folder = actual_folder.parent.parent  # notebooks/clusterizacao -> notebooks -> projeto
path_dataset = main_folder / 'dataset' / 'clusters_datasets' / 'a_traning' / 'X_dataset.csv'

df = pd.read_csv(path_dataset)
print(f'Dataset carregado: {df.shape[0]} amostras, {df.shape[1]} features')

Dataset carregado: 178 amostras, 13 features


In [3]:
#Dataset para treinamento das variaveis
X = df.values

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [4]:
# KMeans default: n_clusters=8
kmeans_def = KMeans(random_state=42)
labels_def = kmeans_def.fit_predict(X)
n_clusters_def = len(set(labels_def))

print(f'Clusters encontrados (default): {n_clusters_def}')

Clusters encontrados (default): 8


## Passo 2 — Performance no treino (default)

In [5]:
sil_def = mt.silhouette_score(X, labels_def)
print(f'Clusters: {n_clusters_def}')
print(f'Silhouette Score: {sil_def:.4f}')

Clusters: 8
Silhouette Score: 0.1882


## Passo 3 — Performance na validação (default)

In [6]:
# Clusterização não possui conjunto de validação separado
print('Clusterização não possui conjunto de validação separado.')
print('Avaliação feita sobre o dataset completo.')

Clusterização não possui conjunto de validação separado.
Avaliação feita sobre o dataset completo.


## Passo 4 — Ajuste de hiperparâmetros

In [7]:
print('--- Ensaio KMeans: variando K ---')
print(f'{"K":>4}  {"Inércia":>12}  {"Silhouette Score":>16}')
print('-' * 38)

best_k, best_ss = 2, -1
for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    lbl = km.fit_predict(X)
    ss = mt.silhouette_score(X, lbl)
    inertia = km.inertia_
    marker = ' <---' if ss > best_ss else ''
    if ss > best_ss:
        best_k, best_ss = k, ss
    print(f'{k:>4}  {inertia:>12.2f}  {ss:>16.4f}{marker}')

print()
print(f'Melhor K: {best_k}  |  Silhouette Score: {best_ss:.4f}')

--- Ensaio KMeans: variando K ---
   K       Inércia  Silhouette Score
--------------------------------------
   2       1017.83            0.2132 <---
   3        829.04            0.2330 <---
   4        750.15            0.2166
   5        681.87            0.1868
   6        622.37            0.2203
   7        568.97            0.2108
   8        533.33            0.1865
   9        505.89            0.1867
  10        477.39            0.1746

Melhor K: 3  |  Silhouette Score: 0.2330


## Passo 5 — União treino + validação

In [8]:
# Dataset de clusterização não possui validação separada — todos os dados já foram usados
print(f'Usando todos os dados para o retreino final: {X.shape}')

Usando todos os dados para o retreino final: (178, 13)


## Passo 6 — Retreino com melhores parâmetros

In [9]:
kmeans_final = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)
labels_final = kmeans_final.fit_predict(X)

print(f'Retreino com K={best_k}')
print(pd.Series(labels_final).value_counts().sort_index().to_string())

Retreino com K=3
0    61
1    63
2    54


## Passo 7 — Performance no teste

In [10]:
sil_final = mt.silhouette_score(X, labels_final)
inertia_final = kmeans_final.inertia_

print(f'--- Performance Final (K={best_k}) ---')
print(f'Silhouette Score : {sil_final:.4f}')
print(f'Inércia (WCSS)   : {inertia_final:.2f}')

--- Performance Final (K=3) ---
Silhouette Score : 0.2330
Inércia (WCSS)   : 829.04


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [11]:
data = {
    'Configuração':     [f'Default (K={n_clusters_def})', f'Tunado (K={best_k})'],
    'N° de Clusters':   [n_clusters_def, best_k],
    'Silhouette Score': [f'{sil_def:.4f}', f'{sil_final:.4f}'],
    'Inércia (WCSS)':   [f'{kmeans_def.inertia_:.2f}', f'{inertia_final:.2f}'],
}
df_comp = pd.DataFrame(data)
print('--- Quadro Comparativo — Diagnóstico Completo ---')
print(df_comp.to_string(index=False))

--- Quadro Comparativo — Diagnóstico Completo ---
 Configuração  N° de Clusters Silhouette Score Inércia (WCSS)
Default (K=8)               8           0.1882         536.79
 Tunado (K=3)               3           0.2330         829.04
